# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a detailed, step-by-step template for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

The FAIR² dataset contains mass spectrometry analysis data of human extracellular vesicles, describing protein abundances, modification statistics, and related details. All references to dataset elements (record sets, fields, columns) use their Croissant schema `@id` for accuracy and reproducibility.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- Dataset DOI: [10.71728/senscience.vq4a-28xa](https://doi.org/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as object, NOT dict!)
metadata = dataset.metadata  # This is an object with attributes.

print(f"{metadata.name}: {metadata.description}\n")
print(f"Dataset @id: {metadata.id}")

## 2. Data Overview
Review record sets, fields, and their `@id`s.

This will help you understand the available data resources for downstream processing and schema-compliant querying.

In [ ]:
# List all record sets and the fields within, using their @id

print("RecordSets in this dataset:")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {record_set.name}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}, name: {field.name}, datatype: {field.data_type if hasattr(field,'data_type') else 'unknown'}")
    else:
        print("  No fields listed in this record set.")
    print()
    # Optionally, list columns if present
    if hasattr(record_set, 'columns') and record_set.columns:
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - Column @id: {column.id}, name: {column.name}, datatype: {column.data_type if hasattr(column,'data_type') else 'unknown'}")


## 3. Data Extraction
Load data from the main record set (`@id`) into a DataFrame for analysis.

We first select a record set (use the `@id` printed above!) – for this dataset, it's often a single main record set. 
If there are multiple record sets, all will be loaded into individual DataFrames.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]
print("All record sets (by @id):", record_set_ids)

# Extract all records by record set, store as individual DataFrames
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from {rs_id}.")

# Show columns in the main record set
main_record_set_id = record_set_ids[0]
print(f"\nColumns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

# Preview the first 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the data:
- Select a numeric field (e.g., 'MW [kDa]' for molecular weight, or 'Normalized Abundance') by its `@id` or DataFrame column.
- Filter, normalize and group the data for further analysis.


In [ ]:
# We'll try to infer some likely field names from the DataFrame, else you may specify by schema @id or column name.
df = dataframes[main_record_set_id]
print("Available columns for EDA:")
print(list(df.columns))

# Define field IDs (change these if different in your dataset)
# For demonstration, let's try 'MW [kDa]' and group by 'Description' if available.
numeric_field = None
for col in df.columns:
    if 'MW' in col or 'abundance' in col.lower() or 'weight' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.select_dtypes(include=['float', 'int']).columns[0]  # fallback
print(f"Using numeric field: {numeric_field}")

# Remove nulls for EDA
eda_df = df.copy()
eda_df = eda_df[eda_df[numeric_field].notnull()]

threshold = eda_df[numeric_field].mean()
filtered_df = eda_df[eda_df[numeric_field] > threshold]
print(f"Filtered rows where {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} records")

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field if available
group_field = None
candidates = ['Description', 'Gene', 'Protein Name', 'Category', 'Group']  # adjust if needed
for col in df.columns:
    if any(term.lower() in col.lower() for term in candidates):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable categorical group field found.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field (e.g., MW [kDa] or normalized abundance).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field} (filtered)")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If we did group by a categorical field, plot the top means
if group_field:
    plt.figure(figsize=(10, 4))
    top_n = 15
    plot_df = grouped_df.head(top_n)
    sns.barplot(y=group_field, x=numeric_field, data=plot_df)
    plt.title(f"Top {top_n} Groups by Mean {numeric_field}")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Using `mlcroissant`, we:
- Loaded and explored the FAIR² mass spectrometry dataset by referencing all entities via their `@id`
- Examined record set and field structure
- Loaded the main record set data for processing
- Filtered, normalized, and grouped protein-level data (e.g., by molecular weight or abundance)
- Visualized major data distributions and highlighted key group means

This workflow can be adapted for in-depth, schema-compliant analyses of other Croissant datasets!